# PGC305 - Tópicos especiais em LLM e Deep Learning

## Definição dos dados

In [ ]:
import torch; import sklearn

# 1. Carregar dados
iris = sklearn.datasets.load_iris()
X = iris.data        # 4 features: sépalas e pétalas
y = (iris.target == 1).astype(float)  # 1 se Versicolor, 0 caso contrário


In [ ]:
from sklearn.model_selection import train_test_split

# 2. Preparar dados para pytorch
X_raw = torch.tensor(iris.data, dtype=torch.float32)
y_raw = torch.tensor((iris.target == 1).astype(float), dtype=torch.float32).view(-1, 1)

# Normalização (Z-score)
mean = X_raw.mean(dim=0)
std = X_raw.std(dim=0)
X_norm = (X_raw - mean) / std

# Separar entre treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_norm.numpy(), y_raw.numpy(), test_size=0.1, random_state=42
)

# Converter de volta para Tensores
X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

## Definição do modelo e treinamento

In [ ]:
# 3. Definir modelo: regressão logística
modelo = torch.nn.Linear(4, 1)  # 4 features → 1 saída (probabilidade de ser Versicolor)

# 4. Definir função de perda e algoritmo de otimização
funcao_perda = torch.nn.BCEWithLogitsLoss()  # combinação de sigmoid + BCE
optimizer = torch.optim.SGD(modelo.parameters(), lr=0.01, momentum=0.9, dampening=0, weight_decay=0, nesterov=False, maximize=False, foreach=None, differentiable=False, fused=None)

## Execução do treinamento

In [ ]:
# 5. Treino (usando apenas X_train e y_train)
for epoch in range(1000):
    optimizer.zero_grad()
    outputs = modelo(X_train)
    loss = funcao_perda(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Época [{epoch+1}/1000], Loss: {loss.item():.4f}")

In [ ]:
##Validando code

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 6. Avaliação nos conjuntos de TREINO e TESTE
modelo.eval()
with torch.no_grad():
    # Avaliação no Treino
    out_train = modelo(X_train)
    pred_train = (torch.sigmoid(out_train) > 0.5).float()
    acc_train = (pred_train == y_train).float().mean()

    # Avaliação no Teste
    out_test = modelo(X_test)
    pred_test = (torch.sigmoid(out_test) > 0.5).float()
    acc_test = (pred_test == y_test).float().mean()

# Cálculo das métricas adicionais para o Teste
y_test_np = y_test.numpy()
pred_test_np = pred_test.numpy()

f1 = f1_score(y_test_np, pred_test_np)
cm = confusion_matrix(y_test_np, pred_test_np)

print(f"Acurácia no Treino: {acc_train.item() * 100:.2f}%")
print(f"Acurácia no Teste: {acc_test.item() * 100:.2f}%")
print(f"F1-Score no Teste: {f1:.4f}")

# Plot da Matriz de Confusão
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Não-Versicolor', 'Versicolor'], yticklabels=['Não-Versicolor', 'Versicolor'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão (Teste)')
plt.show()

print("\nRelatório de Classificação:")
print(classification_report(y_test_np, pred_test_np, target_names=['Não-Versicolor', 'Versicolor']))

In [ ]:
##Calculo do bias e dos pesos

In [ ]:
print("--- Parâmetros do Modelo ---")
# O modelo é uma camada linear: torch.nn.Linear(4, 1)
# weight.data contém os coeficientes para cada uma das 4 features
# bias.data contém o termo de intercepto

pesos = modelo.weight.data.numpy()
vies = modelo.bias.data.numpy()

print(f"Pesos (Weights):\n{pesos}")
print(f"Viés (Bias): {vies[0]:.4f}")

# Relacionando com as features do Iris
features = ['comprimento sépala', 'largura sépala', 'comprimento pétala', 'largura pétala']
for f, w in zip(features, pesos[0]):
    print(f"Influéncia de {f}: {w:.4f}")